# Credit Derivatives Pricing
파생상품 세션에서 마지막으로 다룰 상품은 이자율 파생상품입니다.<br>
이번 세션에서는 이자율 파생상품의 현금흐름과 가격결정 방법론에 대해 공부합니다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize
import scipy.stats as stats

## 1. CDS Pricing
CDS(Credit Default Swap)는 보장 매수자가 가지는 특정 상품의 채무 불이행 위험을 보장 매도자에게 전가하는 파생상품입니다.<br>
보장 매수자는 매도자에게 지급주기마다 일정 프리미엄을 지급하고, 보장 매도자는 해당 상품이 채무 불이행 상태가 되었을 때,<br>
원금 손실액을 매수자에게 지급합니다. 따라서 CDS의 가치 평가를 위해서는 어떤 상품의 부도율을 계산하는 과정이 필요합니다.<br>
부도율은 무차익거래 법칙을 적용하여 계산합니다. 우선 무위험 현물 이자율 곡선과 위험 현물 이자율 곡선을 준비합니다.<br>
무위험 현물 이자율의 주체가 발행한 무이표채권에 N년동안 투자한 경우, 만기에서의 잔고는 다음과 같습니다.
$$NA \times e^{rf_T \times T} = S_f(t) $$
위험 현물 이자율의 주체가 발행한 무이표채권에 N년동안 투자한 경우, 만기에서의 잔고는 다음과 같습니다.
$$NA \times e^{rd_T \times T} = S_d(t)$$
무차익거래 법칙에 따라 다음 등식이 성립합니다.
$$S_f(t) = S_d(t) \times (1-p_d(t)) + R \times NA \times p_d(t) $$
이때, $p_d(t)$는 해당 상품이 $t$ 시점까지 부도가 날 확률, R은 원금 대비 부도시 보전 가능한 금액의 비율을 의미합니다.<br>
$S_f(t) = NA \times e^{r_f(t) \times t}$ 이고, $S_d(t) = NA \times e^{r_d(t) \times t}$ 이므로 식을 정리하면 다음과 같습니다.<br>


$$e^{r_f(t) \times t} = e^{r_d(t) \times t} \times (1-p_d(t)) + R \times p_d(t) $$

$$p_d(t) = \frac{e^{r_d(t) \times t} - e^{r_f(t) \times t}}{e^{r_d(t) \times t} - R}$$

이 $p_d(t)$ 는 엄밀히 말해서 t시점까지 부도가 발생할 확률, 즉 누적부도확률이니 t시점에서 부도가 발생할 확률은 $p_d(t)$ - $p_d(t-dt)$입니다.<br>
이제 무위험 현물이자율은 모든 시점에 대해 $r_f = 0.03$ 이고, 위험 현물이자율은 다음과 같다고 가정합니다.<br>
$r_d(0.00273) = 0.03$, $r_d(0.25) = 0.032$,$r_d(0.5) = 0.033$,  $r_d(1) = 0.034$,  $r_d(1.5) = 0.035$,  $r_d(2) = 0.036$,  $r_d(3) = 0.04$<br>
이를 통해 3년까지 하루 단위로 누적 부도율과 일일 부도율을 구합니다. 이때 회수율 $R$ 은 0.4를 가정합니다.

In [2]:
def linear_interpolate(Target,t,rt):
    if Target<t[0]:
        return rt[0]
    elif Target>t[len(t)-1]:
        return rt[len(t)-1]
    else:
        for i in range(0,len(t)):
            if Target>t[i]:
                break
        leftT = t[i]
        rightT = t[i+1]
        ratio = (Target - t[i])/(t[i+1]-t[i])
        return ratio * rt[i+1] + (1-ratio) * rt[i]

In [3]:
rate = pd.DataFrame(index = ["rf(T)","rd(T)"], columns = [0.00273, 0.25, 0.5, 1, 1.5, 2, 3])

In [4]:
rate.loc["rf(T)",:] = 0.03
rate.loc["rd(T)",0.00273] = 0.03
rate.loc["rd(T)",0.25] = 0.032
rate.loc["rd(T)",0.5] = 0.033
rate.loc["rd(T)",1] = 0.034
rate.loc["rd(T)",1.5] = 0.035
rate.loc["rd(T)",2] = 0.036
rate.loc["rd(T)",3] = 0.04
Recovery = 0.4

In [5]:
Default = pd.DataFrame(index = ["p_D(t)","p_d(t)"], columns = range(1,3*365+1))
Default.columns/=365
for i in range(0,3*365):
    rf = linear_interpolate(float(Default.columns[i]),np.array(rate.columns),np.array(rate.loc["rf(T)"]))
    rd = linear_interpolate(float(Default.columns[i]),np.array(rate.columns),np.array(rate.loc["rd(T)"]))
    Sf = np.exp(rf*(i+1)/365)
    Sd = np.exp(rd*(i+1)/365)
    Default.loc["p_D(t)",Default.columns[i]] = (Sd - Sf) / (Sd- Recovery)
    if i==0:
        Default.loc["p_d(t)",Default.columns[i]] = Default.loc["p_D(t)",Default.columns[i]]
    else:
        Default.loc["p_d(t)",Default.columns[i]] = Default.loc["p_D(t)",Default.columns[i]] - Default.loc["p_D(t)",Default.columns[i-1]]

매일의 현금흐름은 다음과 같습니다.<br>
1. 부도가 발생하지 않은 경우 : 매수자는 매도자에게 하루 치의 프리미엄을 지급
2. 부도가 발생한 경우 : 매수자는 매도자에게 하루 치의 프리미엄을 지급, 매도자는 매수자에게 $NA \times (1-R)$ 지급

이를 이용해 매일의 기대현금흐름을 구하고, 이의 현재가치를 구하면 CDS의 현재가치가 됩니다. <br>
이때 할인금리는 무위험금리를 사용하고, CDS 프리미엄은 연 3%를 가정합니다.

In [6]:
CDSPremium = 0.03
CDSValue = 0.0
NA = 10000
for i in range(0,3*365):
    default_prob = Default.loc["p_d(t)",Default.columns[i]]
    ExpectCF = (1-default_prob) * CDSPremium/365 * NA + default_prob * (CDSPremium/365 - Recovery * NA)
    rf = linear_interpolate(float(Default.columns[i]),np.array(rate.columns),np.array(rate.loc["rf(T)"]))
    CDSValue = ExpectCF * np.exp(-rf * (i+1)/365)

In [7]:
CDSValue

0.10985705938401993

이제 이 CDS의 현재가치가 0이 되게 하는, 즉 두 주체가 평등하게 되는 CDS 프리미엄을 구하는 것이 CDS 가격결정입니다.

In [8]:
def ValuingCDS(CDSPremium, Trf, Rf, Trd, Rd, NA, Recovery, dt, T):
    CDSValue = 0.0
    NT = int(T/dt)
    p_Dt = 0.0
    p_Dt_last = 0.0
    p_dt = 0.0
    for i in range(0,NT):
        T = (i+1)*dt
        rf = linear_interpolate(T,Trf,Rf)
        rd = linear_interpolate(T,Trd,Rd)
        Sf = np.exp(rf*(i+1)/365)
        Sd = np.exp(rd*(i+1)/365)
        p_Dt = (Sd - Sf) / (Sd - Recovery)
        p_dt = p_Dt - p_Dt_last
        ExpectCF = (1-p_dt) * CDSPremium/365 * NA + p_dt * (CDSPremium/365 - Recovery * NA)
        CDSValue = ExpectCF * np.exp(-rf * T)
        p_Dt_last = p_Dt
    return CDSValue

In [9]:
ValuingCDS(0.03, np.array(rate.columns), np.array(rate.loc["rf(T)"]), np.array(rate.columns), np.array(rate.loc["rd(T)"]), NA, Recovery, 1/365, 3)

0.10985705938401993

In [10]:
r = optimize.fsolve(ValuingCDS, 0.03, args = (np.array(rate.columns), np.array(rate.loc["rf(T)"]), np.array(rate.columns), np.array(rate.loc["rd(T)"]), NA, Recovery, 1/365, 3))[0]
print("적절한 CDS Premium은 "+ str(r*100)[:5] +"% 입니다")

적절한 CDS Premium은 2.561% 입니다


## 2. Basket CDS

CDS 중에서는 단일 상품이 아닌 두 개 이상의 상품에 대한 CDS 또한 존재합니다.<br>
Basket CDS라 불리는 이 상품은 준거자산들이 모여있는 준거자산 Pool에서 특정 신용사건이 존재할 경우 부도로 간주하는 상품입니다.<br>
보통 NTD라고 불리는 조건이 붙어 있는데, 이는 Nth To Default의 약자로 Pool에서 N번째 신용사건이 존재할 경우 성립되는 조건입니다.<br>
예를 들어 FTD(First To Default)의 경우 Pool에서 한 개의 자산이라도 부도가 날 경우, 부도로 간주하는 반면,<br>
STD(Second To Default)의 경우 Pool에서 적어도 두 개의 자산이 부도가 나야 부도로 간주합니다.<br>
해당 상품의 경우, 정확한 부도확률 계산이 불가능하기에 우회하여 부도확률을 계산합니다.<br>

## 2-1. Pricing Basket CDS With Monte Carlo Simulation


첫 번째 방법은 몬테카를로 시뮬레이션을 이용하는 것입니다.<br>
매일에 해당하는 부도확률 $p_d$가 있으니, 해당 날짜에 해당하는 난수 $U_i$를 제작, 표준정규분포에서 $P(Z<U_i) < p_d$ 인 경우 그 시점 이후 해당 path는 부도로 정의합니다.<br>
그렇게 각 path마다 부도 여부와 그에 따른 현금흐름을 계산하여, 현재가치화하여 가치를 계산합니다.<br>
다만 준거자산이 둘 이상일 경우, 준거자산마다 부도확률이 다르고, 부도간의 상관관계가 존재할 수 있으니 이를 고려한 시뮬레이션을 진행해야 합니다.

In [11]:
rate2 = pd.DataFrame(index = ["rf(T)","rd(T)","rd2(T)"], columns = [0.00273, 0.25, 0.5, 1, 1.5, 2, 3])
rate2.loc["rf(T)",:] = 0.03
rate2.loc["rd(T)",0.00273] = 0.03
rate2.loc["rd(T)",0.25] = 0.032
rate2.loc["rd(T)",0.5] = 0.033
rate2.loc["rd(T)",1] = 0.034
rate2.loc["rd(T)",1.5] = 0.035
rate2.loc["rd(T)",2] = 0.036
rate2.loc["rd(T)",3] = 0.04
rate2.loc["rd2(T)",0.00273] = 0.03
rate2.loc["rd2(T)",0.25] = 0.0325
rate2.loc["rd2(T)",0.5] = 0.0335
rate2.loc["rd2(T)",1] = 0.0345
rate2.loc["rd2(T)",1.5] = 0.0355
rate2.loc["rd2(T)",2] = 0.0365
rate2.loc["rd2(T)",3] = 0.045

Default2 = pd.DataFrame(index = ["p_D(t)","p_d(t)","p_D2(t)","p_d2(t)","Z(t)","Z2(t)"], columns = range(1,3*365+1))
Default2.columns/=365
for i in range(0,3*365):
    rf = linear_interpolate(float(Default2.columns[i]),np.array(rate2.columns),np.array(rate2.loc["rf(T)"]))
    rd = linear_interpolate(float(Default2.columns[i]),np.array(rate2.columns),np.array(rate2.loc["rd(T)"]))
    rd2 = linear_interpolate(float(Default2.columns[i]),np.array(rate2.columns),np.array(rate2.loc["rd2(T)"]))
    Sf = np.exp(rf*(i+1)/365)
    Sd = np.exp(rd*(i+1)/365)
    Sd2 = np.exp(rd2*(i+1)/365)
    Default2.loc["p_D(t)",Default2.columns[i]] = (Sd - Sf) / (Sd- Recovery)
    Default2.loc["p_D2(t)",Default2.columns[i]] = (Sd2 - Sf) / (Sd2- Recovery)
    if i==0:
        Default2.loc["p_d(t)",Default2.columns[i]] = Default2.loc["p_D(t)",Default2.columns[i]]
        Default2.loc["p_d2(t)",Default2.columns[i]] = Default2.loc["p_D2(t)",Default2.columns[i]]
        Default2.loc["Z(t)",Default2.columns[i]] = stats.norm.ppf(Default2.loc["p_D(t)",Default2.columns[i]])
        Default2.loc["Z2(t)",Default2.columns[i]] = stats.norm.ppf(Default2.loc["p_D2(t)",Default2.columns[i]])
    else:
        Default2.loc["p_d(t)",Default2.columns[i]] = Default2.loc["p_D(t)",Default2.columns[i]] - Default2.loc["p_D(t)",Default2.columns[i-1]]
        Default2.loc["p_d2(t)",Default2.columns[i]] = Default2.loc["p_D2(t)",Default2.columns[i]] - Default2.loc["p_D2(t)",Default2.columns[i-1]]
        Default2.loc["Z(t)",Default2.columns[i]] = stats.norm.ppf(Default2.loc["p_D(t)",Default2.columns[i]] - Default2.loc["p_D(t)",Default2.columns[i-1]])
        Default2.loc["Z2(t)",Default2.columns[i]] = stats.norm.ppf(Default2.loc["p_D2(t)",Default2.columns[i]] - Default2.loc["p_D2(t)",Default2.columns[i-1]])

In [12]:
Default2

,0.002740,0.005479,0.008219,0.010959,0.013699,0.016438,0.019178,0.021918,0.024658,0.027397,...,2.975342,2.978082,2.980822,2.983562,2.986301,2.989041,2.991781,2.994521,2.997260,3.000000
p_D(t),0.0,0.0,0.000001,0.000001,0.000002,0.000003,0.000004,0.000006,0.000007,0.000009,...,0.104697,0.104871,0.105046,0.105221,0.105396,0.105571,0.105746,0.105921,0.106096,0.106272
p_d(t),0.0,0.0,0.0,0.000001,0.000001,0.000001,0.000001,0.000001,0.000002,0.000002,...,0.000174,0.000175,0.000175,0.000175,0.000175,0.000175,0.000175,0.000175,0.000175,0.000175
p_D2(t),0.0,0.0,0.000001,0.000002,0.000003,0.000004,0.000005,0.000007,0.000009,0.000011,...,0.128545,0.128755,0.128966,0.129177,0.129387,0.129598,0.12981,0.130021,0.130232,0.130443
p_d2(t),0.0,0.0,0.000001,0.000001,0.000001,0.000001,0.000002,0.000002,0.000002,0.000002,...,0.00021,0.000211,0.000211,0.000211,0.000211,0.000211,0.000211,0.000211,0.000211,0.000211
Z(t),-6.161986,-5.066396,-4.93293,-4.853238,-4.795942,-4.751058,-4.714092,-4.682628,-4.655215,-4.630912,...,-3.575981,-3.575824,-3.575668,-3.575513,-3.575357,-3.575202,-3.575047,-3.574892,-3.574738,-3.574584
Z2(t),-6.126563,-5.02373,-4.889181,-4.808815,-4.751023,-4.705743,-4.668445,-4.636694,-4.609029,-4.584501,...,-3.526693,-3.52655,-3.526408,-3.526266,-3.526125,-3.525984,-3.525843,-3.525702,-3.525561,-3.525421


In [13]:
NTrial = 10000
rho = 0.5
T = 3
dt = 1/365
Normal1 = np.random.normal(size=(int(T/dt), NTrial))
Normal2 = np.random.normal(size=(int(T/dt), NTrial))
nor1 = Normal1.reshape(1,int(T/dt)*NTrial)
nor2 = Normal2.reshape(1,int(T/dt)*NTrial)
nor2 = rho*nor1 + np.sqrt(1-rho**2)*nor2
Normal1 = nor1.reshape(int(T/dt),NTrial)
Normal2 = nor2.reshape(int(T/dt),NTrial)
IsDefault1 = np.zeros((int(T/dt)+1,NTrial))
IsDefault2 = np.zeros((int(T/dt)+1,NTrial))
IsDefaultFTD = np.zeros((int(T/dt)+1,NTrial))
IsDefaultSTD = np.zeros((int(T/dt)+1,NTrial))

이전에 이미 부도가 나 있거나, 해당 날짜의 난수가 부도 범위에 있을 경우, 해당 path는 해당 시점에서 부도가 났다고 간주합니다.<br>
FTD는 해당 시점에서 두 자산 중 하나라도 부도가 나면 부도로 간주, STD는 두 자산이 모두 부도가 나야 부도로 간주합니다.

In [14]:
IsDefault1 = np.zeros((int(T/dt)+1,NTrial))
IsDefault2 = np.zeros((int(T/dt)+1,NTrial))
IsDefaultFTD = np.zeros((int(T/dt)+1,NTrial))
IsDefaultSTD = np.zeros((int(T/dt)+1,NTrial))
for i in range(1,int(T/dt)+1):
    IsDefault1[i] = ((IsDefault1[i-1] == 1)|(Normal1[i-1]<Default2.loc["Z(t)",Default2.columns[i-1]]))
    IsDefault2[i] = ((IsDefault2[i-1] == 1)|(Normal2[i-1]<Default2.loc["Z2(t)",Default2.columns[i-1]]))
    IsDefaultFTD[i] = ((IsDefault1[i]==1)|(IsDefault2[i]==1))
    IsDefaultSTD[i] = ((IsDefault1[i]==1)&(IsDefault2[i]==1))

이제 각 Path별 부도 날짜를 구한 뒤, 부도 날짜까지 발생할 현금흐름의 현재가치를 구합니다.

In [15]:
FTDPremium = 0.05
STDPremium = 0.01
FTDDefaultT = np.zeros((NTrial))
STDDefaultT = np.zeros((NTrial))
for i in range(0,NTrial):
    for j in range(1,int(T/dt)+1):
        if(IsDefaultFTD[j,i]==1):
            break
    FTDDefaultT[i] = j

for i in range(0,NTrial):
    for j in range(1,int(T/dt)+1):
        if(IsDefaultSTD[j,i]==1):
            break
    STDDefaultT[i] = j

FTDDefaultValue = np.zeros((int(T/dt)))
FTDNormalValue = np.zeros((int(T/dt)))
STDDefaultValue = np.zeros((int(T/dt)))
STDNormalValue = np.zeros((int(T/dt)))
for i in range(0,int(T/dt)):
    rf = linear_interpolate(float(Default2.columns[i]),np.array(rate2.columns),np.array(rate2.loc["rf(T)"]))
    if(i==0):
        FTDNormalValue[i] = (NA * FTDPremium * (i+1)/365) * np.exp(-rf * (i+1)/365)
        STDNormalValue[i] = (NA * STDPremium * (i+1)/365) * np.exp(-rf * (i+1)/365)
    else:
        FTDNormalValue[i] = FTDNormalValue[i-1] + (NA * FTDPremium * 1/365) * np.exp(-rf * (i+1)/365)
        STDNormalValue[i] = STDNormalValue[i-1] + (NA * STDPremium * 1/365) * np.exp(-rf * (i+1)/365)
    FTDDefaultValue[i] = FTDNormalValue[i] - NA * (1-Recovery) * np.exp(-rf * (i+1)/365)
    STDDefaultValue[i] = STDNormalValue[i] - NA * (1-Recovery) * np.exp(-rf * (i+1)/365)

FTDValue = 0
STDValue = 0
for i in range(0,NTrial):
    if (FTDDefaultT[i] == 1095):
        FTDValue += FTDNormalValue[-1]
    else:
        FTDValue += FTDDefaultValue[int(FTDDefaultT[i])]

    if (STDDefaultT[i] == 1095):
        STDValue += STDNormalValue[-1]
    else:
        STDValue += STDDefaultValue[int(STDDefaultT[i])]

FTDValue/=NTrial
STDValue/=NTrial

print("FTD CDS의 현재가치는 "+ str(FTDValue)[:6] +" bp입니다")
print("STD CDS의 현재가치는 "+ str(STDValue)[:6] +" bp입니다")

FTD CDS의 현재가치는 122.59 bp입니다
STD CDS의 현재가치는 204.62 bp입니다
